# DeepGuard — official DF40 download to Google Drive

Uses the official DF40 Google Drive distributions supplied by the dataset authors. The processed testing set is downloaded directly to persistent Google Drive storage. No 110-GB local `/content` staging is used.

Start with the **processed DF40 testing data** (~93 GB, fake data only) and the official JSON metadata. Genuine data for the 31 known methods is handled separately via the official FF++/Celeb-DF distributions.


In [ ]:
!pip -q install -U gdown
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil, subprocess, sys, os, json, time
ROOT=Path('/content/drive/MyDrive/DeepGuard')
DATA=ROOT/'datasets/DF40'
JSON=ROOT/'preprocessing/dataset_json'
DATA.mkdir(parents=True,exist_ok=True); JSON.mkdir(parents=True,exist_ok=True)
usage=shutil.disk_usage('/content/drive')
print('Drive total:',round(usage.total/1e12,2),'TB')
print('Drive free :',round(usage.free/1e12,2),'TB')


## Official DF40 links

These URLs are taken from the DF40 authors' Terms/Download page.


In [ ]:
DF40_TEST_URL='https://drive.google.com/drive/folders/1U8meBbqVvmUkc5GD0jxct6xe6Gwk9wKD?usp=drive_link'
DF40_TRAIN_URL='https://drive.google.com/drive/folders/1980LCMAutfWvV6zvdxhoeIa67TmzKLQ_?usp=drive_link'
DF40_JSON_URL='https://drive.google.com/drive/folders/19VhAL4aDJOKvhl9stEq_ymFeHiXo6_j-?usp=drive_link'
FFPP_REAL_URL='https://drive.google.com/file/d/1dHJdS0NZ6wpewbGA5B0PdIBS9gz28pdb/view?usp=drive_link'
CELEBDF_REAL_URL='https://drive.google.com/file/d/1P9Ep4-nxGpBX8LZGq2UyxqoCL6sBDN8Z/view?usp=drive_link'
print('Official DF40 test folder:',DF40_TEST_URL)


In [ ]:
import subprocess, os, time
def download_folder(url, target):
    target=str(target)
    os.makedirs(target,exist_ok=True)
    cmd=['gdown','--folder',url,'-O',target,'--remaining-ok']
    print('Starting:', ' '.join(cmd))
    return subprocess.run(cmd, check=False).returncode

# This is the only large download in the initial phase. It writes directly to Drive.
rc=download_folder(DF40_TEST_URL, DATA)
print('DF40 test download exit code:',rc)


In [ ]:
# Download the JSON metadata directly to Drive.
rc=download_folder(DF40_JSON_URL, JSON)
print('DF40 JSON download exit code:',rc)


In [ ]:
files=sum(1 for p in DATA.rglob('*') if p.is_file())
bytes_=sum(p.stat().st_size for p in DATA.rglob('*') if p.is_file())
json_files=sum(1 for p in JSON.rglob('*') if p.is_file())
print('DF40 files:',files)
print('DF40 size:',round(bytes_/1e9,2),'GB')
print('JSON files:',json_files)
( ROOT/'logs/df40_download.json').write_text(json.dumps({'timestamp':time.time(),'test_url':DF40_TEST_URL,'data_files':files,'data_bytes':bytes_,'json_files':json_files},indent=2))


## Next stage

Once the processed test set is present, we can run the detector without preprocessing the original videos. For a proper binary LR benchmark we still need genuine/reference data. For the 31 known methods, the DF40 authors point to the official FF++ and Celeb-DF real-data distributions; the 9 unknown methods already include their corresponding real data.
